In [35]:
import os
from pathlib import Path
from datetime import date
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pd_settings import configure_pandas_display
configure_pandas_display()

# ------------------------------
# 1. Setup & File Loading
# ------------------------------
eh = pd.read_csv("eh_sorted.csv", encoding="utf-8-sig")

# ------------------------------
# 2. Rename & Clean Columns
# ------------------------------

# eh.columns = ['title','published','views','likes','comments','video id','duration','url','series name','is series','episode type','episode number']

# eh['title'] = eh['title'].str.replace(r'- Extra History ', '', regex=True)
eh['published'] = pd.to_datetime(eh['published'])
eh['year'] = eh['published'].dt.year.astype(str)

# ------------------------------
# 3. Derivative Columns
# ------------------------------

today = pd.Timestamp.today()
eh['daysSince'] = (today - eh['published']).dt.days
eh['viewsPerDay'] = (eh['views'] / eh['daysSince']).round(0)

# Episode number within series
# eh['epNo'] = eh.groupby('series').cumcount() + 1
eh['seriesLength'] = eh.groupby('series name')['series name'].transform('count')

# Final episode per series
last_eps = eh.groupby('series name')['published'].idxmax()
last_eps_df = eh.loc[last_eps].sort_values('published')
last_eps_df['seriesNo'] = range(1, len(last_eps_df) + 1)

# Merge series number back to main df
eh = eh.merge(last_eps_df[['series name', 'seriesNo']], on='series name', how='left')

# Convert year to ordered category
years_ordered = sorted(eh['year'].unique())
eh['year'] = pd.Categorical(eh['year'], categories=years_ordered, ordered=True)

# ------------------------------
# 4. Summary: Views by Series
# ------------------------------

sum_views = (
    eh.groupby(['seriesNo', 'series name'])
    .agg(seriesViews=('views', 'sum'))
    .reset_index()
    .sort_values('seriesNo')
)

In [36]:
eh_epis = eh[eh['episode type'] == 'Episode'].copy()
eh_epis = eh_epis[eh_epis['episode type'] == 'Episode'].copy()

# have to convert to int(y) b/c ['year'] is a categorical type 
eh_epis["odd even year"] = eh_epis["year"].apply(lambda y: "Even Year" if int(y) % 2 == 0 else "Odd Year")

# print(eh_epis[['title', 'series name', 'episode number']])
# print(eh_epis.columns)

import plotly.express as px

In [ ]:
# Set height dynamically based on number of unique series
n_series = eh_epis["series name"].nunique()
height_per_series = 20
fig_height = n_series * height_per_series

fig = px.scatter(
    eh_epis,
    x="views",
    y="series name",
    color="odd even year",
    hover_name="title"
)

# Ensure all series names show
fig.update_layout(
    height=fig_height,
    yaxis=dict(tickmode='linear')
)

fig.update_yaxes(
    automargin=True,  # Avoid clipping labels
    range=[-0.5, n_series - 0.5],  # Remove excess padding at top/bottom
    categoryorder="array",  # Ensure order is preserved
    categoryarray=eh_epis["series name"].unique()[::-1]  # Reverse order if needed
)

fig.show()

# --- Second Figure: Logarithmic Scale ---
fig_log = px.scatter(
    eh_epis,
    x="views",
    y="series name",
    color="odd even year",
    hover_name="title"
)

fig_log.update_layout(
    title="Views by Series (Log Scale)",
    height=fig_height,
    yaxis=dict(tickmode='linear'),
    xaxis_type='log'
)

fig_log.update_yaxes(
    automargin=True,
    range=[-0.5, n_series - 0.5],
    categoryorder="array",
    categoryarray=eh_epis["series name"].unique()[::-1]
)

fig_log.show()
